# Part 2.4 Backends

After this notebook you can choose where an agent's files live: in the thread, on disk, or
in a Store that outlives the thread.

In [ ]:
from kea.config import MODEL, setup

setup("kea-rebooking-local")

In [ ]:
from uuid import uuid4

from langchain_core.messages import HumanMessage
from langgraph.types import Command


async def ask(agent, text, thread_id=None):
    """Send one message. If the agent pauses for approval, approve and carry on."""
    config = {"configurable": {"thread_id": thread_id or str(uuid4())}}
    result = await agent.ainvoke({"messages": [HumanMessage(text)]}, config)
    while result.get("__interrupt__"):
        requests = result["__interrupt__"][0].value["action_requests"]
        for request in requests:
            print(f"Paused for approval: {request['name']}({request['args']})")
        decisions = [{"type": "approve"}] * len(requests)
        result = await agent.ainvoke(Command(resume={"decisions": decisions}), config)
    return result


def steps(result):
    """Print the tools the agent called, in order."""
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            args = ", ".join(f"{k}={str(v)[:40]!r}" for k, v in call["args"].items())
            print(f"  {call['name']}({args})")


def final(result):
    print(result["messages"][-1].text)

## See it

Deep Agents gives the model file tools. A **backend** decides where those files live. The
finished agent routes three path prefixes to three places:

In [ ]:
from kea.rebooking.agent import build_backend

for prefix, route in build_backend().routes.items():
    print(f"{prefix:<12} -> {type(route).__name__}")
print(f"{'anything else':<12} -> StateBackend")

<img src="../../assets/screenshots/p2-04-state-files.png" alt="The trace of the writefile call that saves the case note /notes/abc123.md into the agent's state." width="900">

## Build it

A `CompositeBackend` routes by path prefix, so one agent gets all three:

| Path | Backend | Lifetime |
|---|---|---|
| anything else | `StateBackend` | this thread |
| `/memory/`, `/skills/` | `FilesystemBackend` on the package files | read from disk |
| `/memories/` | `StoreBackend` | across threads |

In [ ]:
from pathlib import Path

from deepagents import create_deep_agent
from deepagents.backends import (
    CompositeBackend,
    FilesystemBackend,
    StateBackend,
    StoreBackend,
)
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

import kea.rebooking
from kea.rebooking.tools import get_booking

HERE = Path(kea.rebooking.__file__).parent
PREFERENCES = ("kea-air", "passenger-preferences")
store = InMemoryStore()

backend = CompositeBackend(
    default=StateBackend(),
    routes={
        "/memory/": FilesystemBackend(root_dir=str(HERE / "memory")),
        "/skills/": FilesystemBackend(root_dir=str(HERE / "skills")),
        "/memories/": StoreBackend(namespace=lambda rt: PREFERENCES),
    },
)

backend_agent = create_deep_agent(
    model=MODEL,
    tools=[get_booking],
    backend=backend,
    store=store,
    checkpointer=InMemorySaver(),
    system_prompt="You are Kea Air's rebooking assistant.",
)

In [ ]:
result = await ask(
    backend_agent,
    "Look up ABC123 (last name Example). Save a case note to /notes/abc123.md, and a "
    "one-line summary of the cancellation to /memories/cases.md.",
)
print("in this thread's state:", list(result["files"]))
print("in the Store:", [item.key for item in store.search(PREFERENCES)])

## Under the hood

- A route strips its prefix before the backend sees the path. The agent writes
  `/memories/cases.md`; the `StoreBackend` sees `/cases.md`. That is why `/memory/` and
  `/skills/` are separate routes, each rooted at its own directory.
- A Store file is an item whose key is the path and whose value holds the content:

In [ ]:
item = store.search(PREFERENCES)[0]
print(item.namespace, item.key)
print(item.value["content"])

- The `namespace` function receives the runtime and returns the tuple that scopes the
  files. Here it is one fixed tuple. The finished agent uses the caller's identity, so one
  passenger's files are invisible to another. Get this wrong and you have built a data
  leak.
- The finished agent also passes `permissions=[FilesystemPermission(...)]` to deny writes
  under `/memory/` and `/skills/`. The agent can write its memories but not its own rules.
- Sandboxes are backends too. See the [appendix](10_appendix_sandboxes.ipynb).

Docs: [Backends](https://docs.langchain.com/oss/python/deepagents/backends).

## Recap

- A backend decides where the agent's files live. The default is the thread's state.
- A `CompositeBackend` routes path prefixes to different backends.
- Scope the Store namespace per user, and deny writes to the agent's own instructions.

**Next:** [Part 2.5 Memory](05_memory.ipynb)